# Notebook 05 — Distance Setup

**Green Slotting: Phase 1 of the Optimization Stage**

This notebook builds the *aisle-aware distance function* that the Genetic Algorithm
will use in Notebook 06. It answers one question for every storage slot:

> "How far does a picker travel from the warehouse I/O point (dock) to this slot,
> following the aisles — not a straight line?"

**Inputs** (place these in the same folder as this notebook):
- `model_results.csv` — LightGBM demand forecasts (from your ML stage)
- `Storage_Location.csv` — 2,292 slots with (x, y, z) coordinates
- `Support_Points_Navigation.csv` — corridor waypoints (aisle nodes)

**Outputs:**
- `distance_lookup.csv` — every slot with its distance to I/O (the key file for the GA)
- `sp_distance_matrix.npy` — distances between support points
- `support_points_parsed.csv` — cleaned support point coordinates

Run the cells top to bottom.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

print("Libraries loaded.")

In [ ]:
import os as _os
# ── Image output folder ─────────────────────────────────────────────────────
# Figures are saved to a "figures/" subfolder in the directory where you run
# this notebook. Change FIGURES_DIR below if you prefer a different path.
FIGURES_DIR = _os.path.join(_os.getcwd(), "figures")
_os.makedirs(FIGURES_DIR, exist_ok=True)
print(f"[INFO] Figures will be saved to: {FIGURES_DIR}")

## 2. Configuration

Everything you might want to change lives here.

- **`IO_POINT_LABEL`** — the warehouse dock / start point. We use `LC-01`, the
  bottom-left corner (lowest y, leftmost x), which is the standard depot position.
  If your team decides the dock is elsewhere, change it here — nothing else needs editing.
- **`Z_PENALTY`** — how much a vertical (level) move costs relative to a horizontal
  one. `0.5` means going up one rack level costs half a horizontal unit. Reaching a
  higher shelf takes some effort but far less than walking across the floor.

In [ ]:
DATA_DIR = "."            # folder holding the CSV files ("." = same folder as notebook)
IO_POINT_LABEL = "LC-01"  # warehouse dock / start point (bottom-left corner)
Z_PENALTY = 0.5           # vertical move cost relative to horizontal

print(f"I/O point: {IO_POINT_LABEL}")
print(f"Vertical penalty: {Z_PENALTY}")

## 3. Load demand weights (from LightGBM)

Each product is a `Reference_Size` combination. `demand` is LightGBM's predicted
number of picks over the 20-business-day horizon — this becomes the *weight* in the
GA's objective: high-demand items should end up close to the dock.

In [ ]:
demand_df = pd.read_csv(f"{DATA_DIR}/model_results.csv")
demand_df = demand_df[["unique_id", "lgb_pred"]].rename(
    columns={"unique_id": "product_id", "lgb_pred": "demand"})

print(f"Loaded {len(demand_df)} products.")
print(f"Demand range: {demand_df['demand'].min():.2f} to {demand_df['demand'].max():.2f}")
demand_df.head()

## 4. Load storage locations

2,292 physical slots. Each has an `(x, y, z)` coordinate:
- **x** = position across the warehouse width
- **y** = position along the warehouse depth
- **z** = rack level (1 = floor, up to 4)

In [ ]:
loc_df = pd.read_csv(f"{DATA_DIR}/Storage_Location.csv")
loc_df = loc_df[["originalLocation", "x", "y", "z"]].rename(
    columns={"originalLocation": "location_id"})

print(f"Loaded {len(loc_df)} storage locations.")
print(f"x: {loc_df['x'].min()}–{loc_df['x'].max()}, "
      f"y: {loc_df['y'].min()}–{loc_df['y'].max()}, "
      f"z: {loc_df['z'].min()}–{loc_df['z'].max()}")
loc_df.head()

## 5. Parse the support points (corridor waypoints)

`Support_Points_Navigation.csv` has an unusual format — each row is like
`(66.0, -29.0, 1.0);LC-01`, i.e. a coordinate tuple and a label separated by a
semicolon. The helper below extracts the three numbers and the label robustly using
a regex, so odd spacing or trailing `.0`s don't trip it up.

These waypoints are the "intersections" of the aisle network. There are three aisle
columns: **LC** (left, x≈66), **CC** (center, x≈403), and **RC** (right, x≈686).

In [ ]:
def parse_support_points(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.lower().startswith("points_specified"):
                continue
            parts = line.split(";")
            if len(parts) < 2:
                continue
            coords_str, label = parts[0].strip(), parts[1].strip()
            nums = re.findall(r"-?\d+\.?\d*", coords_str)
            if len(nums) >= 3:
                rows.append({"label": label,
                             "x": float(nums[0]),
                             "y": float(nums[1]),
                             "z": float(nums[2])})
    return pd.DataFrame(rows)

sp_df = parse_support_points(f"{DATA_DIR}/Support_Points_Navigation.csv")
print(f"Parsed {len(sp_df)} support points.")
print("Aisle columns:", {lbl[:2] for lbl in sp_df['label']})
sp_df.head()

## 6. Distance between support points (aisle-aware)

We use **Manhattan (rectilinear) distance**, not straight-line. Pickers walk along
aisles and cross-aisles at right angles — they can't cut diagonally through racking.
Manhattan distance (|Δx| + |Δy| + penalty·|Δz|) captures that.

This gives us a 44×44 matrix: the walking distance between every pair of waypoints.

In [ ]:
def manhattan(a, b, z_penalty=Z_PENALTY):
    return abs(a[0]-b[0]) + abs(a[1]-b[1]) + z_penalty*abs(a[2]-b[2])

coords = sp_df[["x", "y", "z"]].values
n = len(sp_df)
sp_dist = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sp_dist[i, j] = manhattan(coords[i], coords[j])

print(f"Support-point distance matrix: {sp_dist.shape}")
print(f"Mean pairwise distance: {sp_dist[sp_dist>0].mean():.1f} units")

## 7. Map each slot to its nearest support point

A storage slot isn't itself a waypoint — it sits somewhere along an aisle. So for each
slot we find the closest support point (its "on-ramp" to the aisle network). The
picker's route is then: **slot → nearest waypoint → (through the network) → I/O**.

In [ ]:
sp_coords = sp_df[["x", "y", "z"]].values
loc_coords = loc_df[["x", "y", "z"]].values

nearest_sp = np.zeros(len(loc_df), dtype=int)
for i, lc in enumerate(loc_coords):
    d = (np.abs(sp_coords[:,0]-lc[0])
         + np.abs(sp_coords[:,1]-lc[1])
         + Z_PENALTY*np.abs(sp_coords[:,2]-lc[2]))
    nearest_sp[i] = d.argmin()

loc_df["nearest_sp_idx"] = nearest_sp
print("Each slot mapped to its nearest aisle waypoint.")
print(f"Waypoints actually used: {len(set(nearest_sp))} of {len(sp_df)}")

## 8. Compute distance from every slot to the I/O point

The total distance for a slot is:

```
dist_to_io  =  [ nearest waypoint → I/O ]   +   [ slot → its nearest waypoint ]
                 (from the 44×44 matrix)          (the short local "on-ramp" hop)
```

The result, `dist_to_io`, is the single number the GA needs per slot.

In [ ]:
# find the I/O waypoint index
mask = sp_df["label"] == IO_POINT_LABEL
if mask.any():
    io_idx = int(sp_df.index[mask][0])
else:
    io_idx = 0
    print(f"WARNING: {IO_POINT_LABEL} not found — falling back to first point.")
print(f"I/O waypoint: {sp_df.iloc[io_idx]['label']} at "
      f"({sp_df.iloc[io_idx]['x']:.0f}, {sp_df.iloc[io_idx]['y']:.0f})")

# local on-ramp hop: slot -> its nearest waypoint
local_hop = np.zeros(len(loc_df))
for i, lc in enumerate(loc_coords):
    sp = sp_coords[nearest_sp[i]]
    local_hop[i] = (abs(sp[0]-lc[0]) + abs(sp[1]-lc[1])
                    + Z_PENALTY*abs(sp[2]-lc[2]))

# full distance to I/O
loc_df["dist_to_io"] = sp_dist[nearest_sp, io_idx] + local_hop

print(f"\nDistance to I/O — min: {loc_df['dist_to_io'].min():.1f}, "
      f"mean: {loc_df['dist_to_io'].mean():.1f}, "
      f"max: {loc_df['dist_to_io'].max():.1f}")
loc_df[["location_id", "x", "y", "z", "dist_to_io"]].head()

## 9. Save outputs

`distance_lookup.csv` is the file Notebook 06 (the GA) will load.

In [ ]:
loc_df.to_csv("distance_lookup.csv", index=False)
np.save("sp_distance_matrix.npy", sp_dist)
sp_df.to_csv("support_points_parsed.csv", index=False)

print("Saved:")
print("  distance_lookup.csv        (slot -> distance to I/O)  <- GA input")
print("  sp_distance_matrix.npy     (44x44 waypoint distances)")
print("  support_points_parsed.csv  (cleaned waypoints)")

## 10. Sanity check

Two quick visuals to confirm the distance model behaves:
- **Left:** a map of all slots colored by distance to I/O. The red star is the dock.
  Slots near it should be dark (close); slots in the far corner should be light (far).
- **Right:** the distribution of distances. A smooth spread from near-dock slots to
  far ones is what we expect.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

sc = ax[0].scatter(loc_df["x"], loc_df["y"], c=loc_df["dist_to_io"],
                   cmap="viridis_r", s=12)
ax[0].scatter(sp_df.iloc[io_idx]["x"], sp_df.iloc[io_idx]["y"],
              c="red", s=200, marker="*", edgecolors="black",
              linewidth=1, label="I/O point", zorder=5)
ax[0].set_title("Slot distance to I/O (darker = closer)")
ax[0].set_xlabel("x (width)"); ax[0].set_ylabel("y (depth)")
ax[0].legend()
plt.colorbar(sc, ax=ax[0], label="distance")

ax[1].hist(loc_df["dist_to_io"], bins=40, color="steelblue", edgecolor="black")
ax[1].axvline(loc_df["dist_to_io"].mean(), color="red", linestyle="--",
              label=f"mean = {loc_df['dist_to_io'].mean():.0f}")
ax[1].set_title("Distribution of slot distances")
ax[1].set_xlabel("distance to I/O"); ax[1].set_ylabel("number of slots")
ax[1].legend()

plt.tight_layout()
plt.savefig(_os.path.join(FIGURES_DIR, "05_slot_distance_map.png"), dpi=150, bbox_inches="tight")
plt.show()

## Done — Phase 1 complete ✓

You now have `distance_lookup.csv`: for all 2,292 slots, the aisle-aware distance a
picker travels from the dock. This is the foundation the Genetic Algorithm builds on.

**Next:** Notebook 06 — the GA reads this file plus the demand weights and searches
for the product→slot assignment that minimizes total weighted travel.

In [ ]:
# ── Extra graphic: Distance-to-I/O distribution by rack level ─────────────
import matplotlib.pyplot as plt
import os as _os

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Boxplot by z-level
level_data = [loc_df[loc_df["z"] == z]["dist_to_io"].values for z in sorted(loc_df["z"].unique())]
bp = axes[0].boxplot(level_data, patch_artist=True,
                     labels=[f"Level {int(z)}" for z in sorted(loc_df["z"].unique())])
colors_lvl = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
for patch, color in zip(bp["boxes"], colors_lvl):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_title("Distance to I/O by Rack Level", fontweight="bold")
axes[0].set_ylabel("Distance (units)")
axes[0].grid(alpha=0.3, axis="y")

# Top-20 nearest vs farthest slots highlighted on the floor map
top20_near = loc_df.nsmallest(20, "dist_to_io")
top20_far  = loc_df.nlargest(20, "dist_to_io")
axes[1].scatter(loc_df["x"], loc_df["y"], c="#CCCCCC", s=8, label="Other slots")
axes[1].scatter(top20_near["x"], top20_near["y"], c="#1d9e75", s=60, label="20 nearest", zorder=5)
axes[1].scatter(top20_far["x"],  top20_far["y"],  c="#C44E52", s=60, label="20 farthest", zorder=5)
axes[1].scatter([sp_df.iloc[io_idx]["x"]], [sp_df.iloc[io_idx]["y"]],
                c="gold", s=250, marker="*", edgecolors="black", linewidth=1,
                label="I/O point", zorder=6)
axes[1].set_title("Nearest & Farthest 20 Slots (floor view)", fontweight="bold")
axes[1].set_xlabel("x"); axes[1].set_ylabel("y")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(_os.path.join(FIGURES_DIR, "05_distance_by_level_and_highlights.png"),
            dpi=150, bbox_inches="tight")
plt.show()